# 41d · **Bậc 2** — cho ConvNeXt thêm kiến trúc, ba arm trên **một** forward

## Bậc 1 để lại gì

```
CNN-ft (dau doc DON GIAN)  0.2202    ti le do manh 0.838 so duong ong 0.2627
corr voi mldec             0.7137    DAT cong (< 0.75), con xa MOC 0.8236
blend w=0.25               +0.0027   duoi cong +0.005, nhung DUONG va tail +0.0011
trong so DEU               -0.0015   -> 41a do -0.0035 la vi dung trong so deu, khong phai
                                        vi huong nay vo dung
```

Thành viên **yếu** là ràng buộc, không phải `corr`. Bậc 2 làm nó mạnh lên.

## Ba arm — **lựa chọn thay thế**, không phải ba thành viên

```
cnn_mld       CNN + ML-Decoder            co lap don bay ML-Decoder (35/37 do +0.01..0.02)
cnn_mld_img   + Tier1Img (phim truoc)     them tin hieu ANH        (53 do G_t1prior +0.0028)
cnn_full      + Tier1 (EHR)               kien truc day du
```

Chúng trả lời câu hỏi *"cho CNN bao nhiêu kiến trúc thì blend tốt nhất"*. Thêm `Tier1` làm thành
viên **mạnh hơn**, nhưng cũng nhập **chính tín hiệu EHR đường ống đã có** ⇒ `corr` tăng. Hai lực
ngược chiều và **không biết cái nào thắng** — nên đo, không đoán.

Ba arm **dùng chung một forward backbone** (`AMENDMENTS ①`: tham số các arm rời nhau ⇒ huấn luyện
n arm trong một vòng lặp **tương đương toán học tuyệt đối** với n lần chạy riêng).

## ConvNeXt cho bao nhiêu token

`convnextv2_nano` @518, stride 32 ⇒ feature map `17×17 = 289` token, dim 640. ML-Decoder và
`Tier1Img` không quan tâm số token K/V — chỉ đổi `D_CXR 768→640`, `N_TOK 1369→289`.

## Chọn và chấm — khoá trước

```
1. ensemble EPOCH trong tung arm        giao thuc 58, cua so `all`
2. chon ARM tren LAT CHON               ba arm canh tranh, lay MOT
3. blend voi DUONG ONG, w tren LAT CHON moc = tpb 0.2627 (KHONG phai probe 0.2042)

cong  blend >= +0.005 macro VA tail khong tut qua -0.003
```

Kèm phép đo phụ miễn phí: **trộn cả ba arm CNN** rồi mới blend — chúng dùng chung backbone nên kỳ
vọng nhỏ, nhưng ensemble epoch cũng "chỉ khác chút" mà cho `+0.005`, nên đo thì biết.

## Ghi đĩa

`/content` là nơi ghi **chính**, Drive chỉ là bản sao — DriveFS đã nuốt ghi 6 lần trong dự án, kể
cả nuốt cả thư mục giữa hai lần `np.save`.

## C0 · Config

In [ ]:
import os, json, math, time, re, glob, datetime, shutil, tarfile, io as _io
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    import timm
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'timm']); import timm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

# Chan spam `Exception ignored in __del__: can only test a child process` cua DataLoader.
# No KHONG di qua `warnings` — Python bao cao qua `unraisablehook`. Loc DUNG loai nay.
import sys as _sys
_old_unraisable = _sys.unraisablehook
def _quiet_unraisable(u):
    if isinstance(u.exc_value, AssertionError) and 'child process' in str(u.exc_value): return
    _old_unraisable(u)
_sys.unraisablehook = _quiet_unraisable

RUN_TAG = 'r1'
RES_LOC = '/content/nb41d'; os.makedirs(RES_LOC, exist_ok=True)   # NOI GHI CHINH
RES41D  = os.path.join(OUT_DIR, 'stage4_nb41d', RUN_TAG)          # ban sao tren Drive
RES41C  = os.path.join(OUT_DIR, 'stage4_nb41c')
SEARCH  = [RES_LOC, RES41D]
def ensure_res():
    try: os.makedirs(RES41D, exist_ok=True)
    except Exception: pass                       # mount hong -> ke, /content moi la noi ghi chinh
    return RES41D
def put(name, fn):
    '''Ghi LOCAL truoc (chac chan), roi sao len Drive. KHONG bao gio ghi thang len mount.'''
    lp = os.path.join(RES_LOC, name); fn(lp)
    try:
        ensure_res(); shutil.copy(lp, os.path.join(RES41D, name))
        if not os.path.exists(os.path.join(RES41D, name)): raise OSError('file khong hien ra')
    except Exception as e:
        print(f'  !! sao {name} len Drive that bai ({type(e).__name__}); ban local o {lp}')
    return lp
def find(pat):
    seen = {}
    for d in SEARCH:
        for f in sorted(glob.glob(os.path.join(d, pat))): seen.setdefault(os.path.basename(f), f)
    return [seen[k] for k in sorted(seen)]
ensure_res()
LOCAL = '/content/cxrlt_518'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')

SRC_PIPE = [os.path.join(OUT_DIR, 'stage4_nb57', 'r3'),
            os.path.join(OUT_DIR, 'stage4_nb53', 'r3')]
ARM_PIPE = ('tpb', 't1_prior_both')
ARM_IMG  = 'mldec'                               # arm ANH-THUAN -> corr backbone thuan

MODEL = 'convnextv2_nano.fcmae_ft_in22k_in1k'
P_BB  = os.path.join(RES41C, 'convnextv2_padchest.pt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
SEED, SEEDS = 42, (42,)                          # bac 2 van la CONG; them seed o bac 3
SIZE, BATCH, ACCUM = 518, 48, 6                  # hieu dung 288 nhu bac 1; 2 anh/mau nen batch nho hon
EPOCHS, LR, WD, WARMUP, CLIP = 8, 1e-4, 0.05, 0.05, 5.0
NW = min(16, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
MOC, GATE_BLEND = 0.8236, 0.005

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

_bb_loc = '/content/convnextv2_padchest.pt'
BB_USE = P_BB if os.path.exists(P_BB) else (_bb_loc if os.path.exists(_bb_loc) else None)
assert BB_USE, f'khong thay trong so PadChest o CA {P_BB} lan {_bb_loc} -> chay 41c C3 truoc'
for p, hd in ((P_OOF, '52a'), (os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet'), '29c'),
              (os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json'), '29c')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'
print(f'trong so CNN: {BB_USE}')

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL     # CUNG cach cat nhu 35..59
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'{MODEL} | E={EPOCHS} LR={LR} | BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

## C1 · EHR + phim trước + ảnh + dự đoán đường ống

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')).set_index('study_id')
_VOC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')))
V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)
print(f'FEAT {FEAT.shape} | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

from PIL import Image
os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh dang co: {_n:,} / {len(ORDER):,}', flush=True)
if _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    _kt = sorted(x for x in os.listdir(READY_X) if x.endswith('.tar'))
    print(f'kho co {len(_kt)} tar -> hydrate', flush=True)
    for _i, f in enumerate(_kt):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                # copy TRUOC — khong mo tar tren mount FUSE
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
        if (_i + 1) % 50 == 0: print(f'  hydrate {_i+1}/{len(_kt)}', flush=True)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, (f'moi co {_n:,}/{len(ORDER):,} anh -> chay C1 cua 41b de stage '
                                  'tu shard goc, hoac kiem kho cxrlt_518_ready')
print(f'anh local: {_n:,} / {len(ORDER):,}')

def load_pipe():
    '''Ensemble TAT CA epoch x seed cho mot arm — giao thuc 58.'''
    out = {}
    for arm in list(ARM_PIPE) + [ARM_IMG]:
        for d in SRC_PIPE:
            fv = sorted(glob.glob(os.path.join(d, f'p_{arm}_s*_ep*.npy')))
            fs = sorted(glob.glob(os.path.join(d, f'psl_{arm}_s*_ep*.npy')))
            if not (fv and fs): continue
            pv = np.mean([np.load(f).astype('float32') for f in fv], 0)
            ps = np.mean([np.load(f).astype('float32') for f in fs], 0)
            assert pv.shape == (len(I_VA), C) and ps.shape == (len(I_SL), C)
            out[arm] = dict(val=pv, sel=ps, n=len(fv)); break
    return out
PIPE = load_pipe()
assert PIPE, f'khong nap duoc du doan duong ong -> doi SRC_PIPE/ARM_PIPE cho khop {SRC_PIPE}'
PIPE_ARM = next(a for a in ARM_PIPE if a in PIPE)
for a, v in PIPE.items(): print(f'  {a:<16} {v["n"]:>3} file')
print(f'  -> blend vao `{PIPE_ARM}` | corr backbone do voi `{ARM_IMG}`')

## C2 · Kiến trúc — `Tier1`/`Tier1Img`/ML-Decoder verbatim `53` r3, backbone ConvNeXt

`convnextv2_nano` @518 cho `17×17 = 289` token dim 640. `forward_features` trả `[B, H, W, Cf]`
(channels-last ở ConvNeXt của `timm`) — cổng 1 kiểm hình dạng thay vì tin giả định.

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    def __init__(s, V, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w): return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    def __init__(s, d_in, d=256, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.mb = MicroBag(V, d) if V else None
        if s.mb is not None: s.bln = nn.LayerNorm(d)
    def forward(s, e, mids=None, mcnt=None):
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([base, s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = patch, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu (38d).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Ban r3: LayerScale `gamma` 1e-4 — chan cu sup ma zero-init khong chan duoc (0/16 epoch-arm
       o 57, truoc do sup 3 lan lien tiep).'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d); s.gamma = nn.Parameter(torch.full((d,), 1e-4)); s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)

class MLDecoder(nn.Module):
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)   # query lop KHONG hoc (bai goc)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, _ = s.img(q, kv); return s.out(h)

class ASL(nn.Module):
    '''Verbatim nb11..59. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y_, P_: float(np.nanmean(ap_cols(Y_, P_)))
tie = lambda Y_, P_: {t: float(np.nanmean(ap_cols(Y_, P_)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn')

_m0 = timm.create_model(MODEL, pretrained=True, num_classes=0, in_chans=1)
_cfg = timm.data.resolve_model_data_config(_m0)
MU, SD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))
_IN_SD = {k: v.clone() for k, v in _m0.state_dict().items()}; del _m0

def build_backbone():
    bb = timm.create_model(MODEL, pretrained=False, num_classes=0, in_chans=1)
    ck = torch.load(BB_USE, map_location='cpu', weights_only=False)
    miss = bb.load_state_dict({k: v for k, v in ck['sd'].items() if not k.startswith('head.')},
                              strict=False)
    bad = [k for k in miss.missing_keys if not k.startswith('head.')]
    assert not bad, f'thieu khoa khi nap backbone PadChest: {bad[:5]}'
    return bb, ck

_bb, _ck0 = build_backbone()
with torch.no_grad():
    _d = max((_IN_SD[k] - v).abs().max().item() for k, v in _bb.state_dict().items() if k in _IN_SD)
assert _d > 1e-4, (f'trong so GIONG HET ImageNet (lech {_d:.1e}) -> `convnextv2_padchest.pt` khong '
                   'duoc nap. Moi so ve do manh/corr se vo nghia.')
# cong 2: hinh dang feature map — KHONG tin gia dinh ve channels-first/last
_bb = _bb.to(dev).eval()
with torch.no_grad():
    _fm = _bb.forward_features(torch.zeros(2, 1, SIZE, SIZE, device=dev))
_shape = tuple(_fm.shape)
_CHLAST = _shape[-1] != _shape[-2]                # [B,H,W,Cf] neu chieu cuoi khac chieu ke cuoi
D_CXR = _shape[-1] if _CHLAST else _shape[1]
N_TOK = (_shape[1]*_shape[2]) if _CHLAST else (_shape[2]*_shape[3])
print(f'cong 2: feature map {_shape} -> {"channels-LAST" if _CHLAST else "channels-FIRST"} | '
      f'{N_TOK} token, d={D_CXR}')
assert 100 <= N_TOK <= 2000 and 256 <= D_CXR <= 2048, f'hinh dang bat thuong: {_shape}'
def to_tokens(fm):
    '''[B,H,W,Cf] hoac [B,Cf,H,W] -> [B, N, Cf]. Quyet dinh bang HINH DANG do duoc o cong 2.'''
    return fm.flatten(1, 2) if _CHLAST else fm.flatten(2).transpose(1, 2)
print(f'cong 3: trong so PadChest da nap | lech ImageNet {_d:.3f} | '
      f'ep {_ck0.get("ep", "?")} mAP PadChest {_ck0.get("mAP", float("nan")):.4f}')
del _bb; torch.cuda.empty_cache()

## C3 · Ba arm + cổng khoá trước

In [ ]:
ARMS = {
  'cnn_mld':     dict(img=False, ehr=False),
  'cnn_mld_img': dict(img=True,  ehr=False),
  'cnn_full':    dict(img=True,  ehr=True),
}
def build_arms(seed=SEED):
    bb, _ = build_backbone(); bb = bb.to(dev)
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)                  # gieo lai TRUOC TUNG arm -> khoi tao giong nhau
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, V=V_MIC)
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
    return bb, M.to(dev), E.to(dev), I.to(dev)

BB, MOD, ETK, IMG = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<14}{"tham so dau doc":>16}')
for t in ARMS: print(f'  {t:<12}{NP_[t]:>16,}')
print(f'  backbone dung chung: {sum(p.numel() for p in BB.parameters()):,} (CO huan luyen)')

_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
assert not MOD['cnn_mld'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    for t in IMG:
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
        assert (IMG[t](_p, _pp, _h0) - _p).abs().max() < 1e-6, f'{t}: has=0 KHONG cat duoc nhanh'
    m = MOD['cnn_full'].eval(); et = ETK['cnn_full'](_e, _mi, _mc)
    l1 = m(_p, et); _k = m.t1; m.t1 = None; l2 = m(_p, et); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
print('cong 4: Tier1 zero-init | Tier1Img LayerScale + has=0 cat sach OK')
BB, MOD, ETK, IMG = build_arms()

## C4 · Huấn luyện — một forward backbone cho **cả ba** arm

Backbone tính **một lần** mỗi batch (hai lần nếu có arm dùng phim trước) rồi ba arm dùng chung.
`56` mắc lỗi gọi lại forward cho từng arm và chạy 15 img/s; ở đây tránh hẳn.

Backbone **có huấn luyện** (khác `53`/`57` nơi RAD-DINO đông cứng) — ConvNeXt rẻ hơn 11 lần nên
khả thi, và `57` đã cho thấy backbone đông cứng không thích nghi thêm được gì.

In [ ]:
from torchvision import transforms as T2
crit = ASL(gn=4, gp=0).to(dev)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])
NEED_PREV = any(c['img'] for c in ARMS.values())

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if (NEED_PREV and pr >= 0)
                                         else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i, pos=k,
                    y=torch.from_numpy(YM[i]))

def mk_loader(idx, train, bs, nw=NW):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=train, num_workers=nw,
                      pin_memory=True, drop_last=train, persistent_workers=nw > 0,
                      prefetch_factor=4 if nw > 0 else None)
_DLC = {}
def eval_loader(idx, bs):
    '''Tao MOT LAN roi dung lai — tao moi moi epoch se ngap spam teardown va ton thoi gian.'''
    k = (len(idx), int(idx[0]), int(idx[-1]), bs)
    if k not in _DLC: _DLC[k] = mk_loader(idx, False, bs)
    return _DLC[k]

def arm_forward(t, p, pp, row, has):
    cfg = ARMS[t]
    q = IMG[t](p, pp, has) if cfg['img'] else p
    et = (ETK[t](FEAT_T[row].to(dev), MIDS_T[row].to(dev), MCNT_T[row].to(dev))
          if cfg['ehr'] else None)
    return MOD[t](q, et)

@torch.no_grad()
def predict(idx, bs=128):
    BB.eval(); MOD.eval(); ETK.eval(); IMG.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(eval_loader(idx, bs), desc='eval', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
            pp = (to_tokens(BB.forward_features(b['image_prev'].to(dev, non_blocking=True))).float()
                  if NEED_PREV else p)
        hs = b['has'].to(dev)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, p, pp, b['row'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

_old = find('epoch_log.json')
LOG = json.load(open(_old[0])) if _old else {}
# Doi tap arm roi resume len nhat ky cu -> bang ket qua tron hai cau hinh ma khong bao gi.
for _sk, _v in LOG.items():
    _eps = [k for k in _v if k.isdigit()]
    if _eps:
        _old_arms = list(_v[_eps[0]])
        assert set(_old_arms) == set(ARMS), (
            f'nhat ky co {len(_old_arms)} arm {_old_arms}, notebook co {len(ARMS)} '
            f'{list(ARMS)}. DOI RUN_TAG o C0 (vd r2) roi chay lai. KHONG xoa gi.')
for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    BB, MOD, ETK, IMG = build_arms(seed)
    PARAM = [p for m in (MOD, ETK, IMG) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': [p for p in PARAM if p.ndim > 1], 'weight_decay': WD, 'lr': LR*10},
         {'params': [p for p in PARAM if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR*10},
         {'params': [p for p in BB.parameters() if p.ndim > 1], 'weight_decay': WD, 'lr': LR},
         {'params': [p for p in BB.parameters() if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR}])
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    dl = mk_loader(I_TF, True, BATCH)
    LOG.setdefault(sk, {})
    for ep in range(EPOCHS):
        BB.train(); MOD.train(); ETK.train(); IMG.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(dl, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
                pp = (to_tokens(BB.forward_features(
                          b['image_prev'].to(dev, non_blocking=True))).float()
                      if NEED_PREV else p)
            y = b['y'].to(dev); hs = b['has'].to(dev); loss = 0.0
            for t in ARMS:
                lg = arm_forward(t, p, pp, b['row'], hs)
                if not torch.isfinite(lg).all(): continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                gn = torch.nn.utils.clip_grad_norm_(PARAM + list(BB.parameters()), CLIP,
                                                    error_if_nonfinite=False)
                if not torch.isfinite(gn):       # total_norm = nan se BOI BAN toan bo tham so
                    opt.zero_grad(set_to_none=True); continue
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        PV, PS = predict(I_VA), predict(I_SL)
        for t in ARMS:
            put(f'p_{t}_{sk}_ep{ep+1}.npy', lambda q, v=PV[t]: np.save(q, v.astype('float16')))
            put(f'psl_{t}_{sk}_ep{ep+1}.npy', lambda q, v=PS[t]: np.save(q, v.astype('float16')))
        # Tier1Img da lam SUP arm 3 lan (53 imgz, 56 tpb, 57 r2 tpb). LayerScale chan duoc o
        # 57 (0/16 epoch-arm) nhung van phai THEO DOI: `ratio_img` = ‖phan du‖/‖patch‖ la dai
        # luong vuot ~1 thi sup; `pr_norm` chi la so gian tiep.
        with torch.no_grad():
            _bd = next(iter(eval_loader(I_VA[:64], 64))); _ri = {}
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                _pb = to_tokens(BB.forward_features(_bd['image'].to(dev))).float()
                _ppb = to_tokens(BB.forward_features(_bd['image_prev'].to(dev))).float()
            _hs2 = _bd['has'].to(dev)
            for t in IMG:
                _ri[t] = float(((IMG[t](_pb, _ppb, _hs2) - _pb).norm(dim=-1)
                                / (_pb.norm(dim=-1) + 1e-9)).mean())
        LOG[sk][str(ep)] = {t: dict(mAP_anh=mac(YM[VA], PV[t]), sel=mac(YM[SL], PS[t]),
                                    pr_norm=(float(IMG[t].pr.weight.norm()) if t in IMG else 0.0),
                                    ratio_img=_ri.get(t, 0.0), **tie(YM[VA], PV[t]))
                            for t in ARMS}
        put('epoch_log.json', lambda q: json.dump(LOG, open(q, 'w')))
        print(f'{sk} ep{ep+1:>2}/{EPOCHS} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph')
        print('   VAL      ' + ' '.join(f'{t}:{mac(YM[VA], PV[t]):.4f}' for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{t}:{mac(YM[SL], PS[t]):.4f}' for t in ARMS))
        print('   TAIL     ' + ' '.join(f'{t}:{tie(YM[VA], PV[t])["tail"]:.4f}' for t in ARMS))
        if _ri:
            print('   ratio_img ' + ' '.join(f'{t}:{v:.4f}' for t, v in _ri.items())
                  + '   (vuot ~1 = phan du AT patch -> sup)')
    put(f'cnn_arch_{sk}.pt', lambda q: torch.save(
        dict(bb=BB.state_dict(), mod=MOD.state_dict(), etk=ETK.state_dict(),
             img=IMG.state_dict(), seed=seed), q))
    LOG[sk]['done'] = True; put('epoch_log.json', lambda q: json.dump(LOG, open(q, 'w')))
print('huan luyen xong')

## C5 · Chọn arm trên lát chọn → blend → cổng

In [ ]:
from scipy.stats import rankdata
def corr(A, B):
    ra, rb = rankdata(A, axis=0), rankdata(B, axis=0)
    ra = (ra - ra.mean(0)) / (ra.std(0) + 1e-12); rb = (rb - rb.mean(0)) / (rb.std(0) + 1e-12)
    return float(np.mean((ra * rb).mean(0)))

CN = {}
for t in ARMS:
    fv, fs = find(f'p_{t}_s*_ep*.npy'), find(f'psl_{t}_s*_ep*.npy')
    if not (fv and fs): continue
    CN[t] = dict(val=np.mean([np.load(f).astype('float32') for f in fv], 0),
                 sel=np.mean([np.load(f).astype('float32') for f in fs], 0), n=len(fv))
assert CN, f'khong thay du doan CNN trong {SEARCH} -> chay C4 truoc'
PV, PS = PIPE[PIPE_ARM]['val'], PIPE[PIPE_ARM]['sel']
M_PIPE, T_PIPE = mac(YM[VA], PV), tie(YM[VA], PV)['tail']
WG = np.round(np.arange(0, 1.001, 0.05), 2)

def blend(cv, cs):
    sc = [mac(YM[SL], (1-w)*PS + w*cs) for w in WG]      # chon w tren LAT CHON
    w = float(WG[int(np.argmax(sc))]); bv = (1-w)*PV + w*cv
    return w, mac(YM[VA], bv) - M_PIPE, tie(YM[VA], bv)['tail'] - T_PIPE, max(sc)

print(f'moc duong ong `{PIPE_ARM}`: {M_PIPE:.4f} | tail {T_PIPE:.4f}\n')
print(f'{"arm":<14}{"n":>3}{"mAP":>9}{"tail":>9}{"corr":>8}{"w":>6}{"d macro":>10}{"d tail":>9}'
      f'{"sel blend":>11}')
ROWS = []
for t, v in CN.items():
    w, dm, dt, sb = blend(v['val'], v['sel'])
    c = corr(v['val'], PIPE[ARM_IMG]['val']) if ARM_IMG in PIPE else corr(v['val'], PV)
    ROWS.append(dict(arm=t, n=v['n'], mAP=mac(YM[VA], v['val']), tail=tie(YM[VA], v['val'])['tail'],
                     corr=c, w=w, d_macro=dm, d_tail=dt, sel_blend=sb))
    print(f'  {t:<12}{v["n"]:>3}{ROWS[-1]["mAP"]:>9.4f}{ROWS[-1]["tail"]:>9.4f}{c:>8.4f}'
          f'{w:>6.2f}{dm:>+10.4f}{dt:>+9.4f}{sb:>11.4f}')
# phep do phu: tron CA BA arm CNN roi moi blend
if len(CN) > 1:
    _av = np.mean([v['val'] for v in CN.values()], 0); _as = np.mean([v['sel'] for v in CN.values()], 0)
    w, dm, dt, sb = blend(_av, _as)
    ROWS.append(dict(arm='TRON 3 ARM', n=sum(v['n'] for v in CN.values()), mAP=mac(YM[VA], _av),
                     tail=tie(YM[VA], _av)['tail'],
                     corr=corr(_av, PIPE[ARM_IMG]['val']) if ARM_IMG in PIPE else corr(_av, PV),
                     w=w, d_macro=dm, d_tail=dt, sel_blend=sb))
    r = ROWS[-1]
    print(f'  {"TRON 3 ARM":<12}{r["n"]:>3}{r["mAP"]:>9.4f}{r["tail"]:>9.4f}{r["corr"]:>8.4f}'
          f'{r["w"]:>6.2f}{r["d_macro"]:>+10.4f}{r["d_tail"]:>+9.4f}{r["sel_blend"]:>11.4f}')

T = pd.DataFrame(ROWS)
# --- CONG SUP: mot arm da sup thi so blend cua no vo nghia (bai hoc 56: doi chung sup lam
#     moi hieu so bi thoi len). Loai truoc khi chon.
_lg = LOG[f's{SEEDS[0]}']; _lastep = str(max(int(k) for k in _lg if k.isdigit()))
_sup = [t for t in CN if _lg[_lastep].get(t, {}).get('ratio_img', 0.0) > 0.9
        or _lg[_lastep].get(t, {}).get('tail', 1.0) < 0.02]
if _sup:
    print(chr(10) + f'!! ARM SUP (ratio_img > 0.9 hoac tail < 0.02): {_sup} -> LOAI khoi lua chon')
    T = T[~T.arm.isin(_sup)]
assert len(T), 'MOI arm deu sup -> khong doc duoc gi. Xem `ratio_img` tung epoch trong epoch_log.'
BEST = T.loc[T.sel_blend.idxmax()]                 # CHON tren lat chon, khong tren val
print(f'\nchon tren LAT CHON -> `{BEST.arm}` (sel blend {BEST.sel_blend:.4f})'
      f'   | tren val se chon `{T.loc[T.d_macro.idxmax()].arm}` — KHONG duoc dung de chon')
print('\n' + '=' * 74)
print(f'CONG: blend >= +{GATE_BLEND} macro VA tail >= -0.003')
_v = ('NHAN -> bac 3: them seed cho arm nay roi chot'
      if (BEST.d_macro >= GATE_BLEND and BEST.d_tail >= -0.003) else
      'VUNG XAM: duoi cong nhung DUONG -> khong CONG BO nhu mot hieu ung, van DUNG duoc trong '
      'ensemble cuoi neu khong am tren lat chon'
      if BEST.d_macro > 0 else 'DONG huong ensemble CNN')
print(f'  `{BEST.arm}` w={BEST.w:.2f}  macro {BEST.d_macro:+.4f} | tail {BEST.d_tail:+.4f} -> {_v}')
print('=' * 74)
print(f'bac 1 (dau doc don gian) do: CNN 0.2202 | corr 0.7137 | blend +0.0027')

L = [f'# REPORT · `nb41d` bac 2 — ConvNeXt + kien truc, ba arm', '',
     f'`{MODEL}` (pretrain PadChest) · backbone CO huan luyen · `{EPOCHS}` epoch · '
     f'seeds `{list(SEEDS)}` · {N_TOK} token d={D_CXR}.', '',
     '| arm | mAP | tail | corr | w | Δmacro | Δtail |', '|---|---|---|---|---|---|---|']
for r in ROWS:
    L.append(f'| `{r["arm"]}` | {r["mAP"]:.4f} | {r["tail"]:.4f} | {r["corr"]:.4f} | '
             f'{r["w"]:.2f} | {r["d_macro"]:+.4f} | {r["d_tail"]:+.4f} |')
L += ['', f'Moc duong ong `{PIPE_ARM}` = {M_PIPE:.4f} / tail {T_PIPE:.4f}.',
      f'Chon tren lat chon: **`{BEST.arm}`**. Ket luan cong: **{_v}**', '',
      '## Nhung gi ket qua nay KHONG tra loi', '',
      '- **Mot seed** — bac 2 la cong. Bac 3 them seed roi moi chot.',
      '- `corr` do voi arm ANH-THUAN cua duong ong; blend do voi arm DAY DU.',
      '- Ba arm dung chung backbone nen `TRON 3 ARM` khong phai ensemble di chung that su.']
_rp = put('REPORT_STAGE4_NB41D.md',
          lambda q: open(q, 'w', encoding='utf-8').write(chr(10).join(L)))
put('blend_table.csv', lambda q: T.to_csv(q, index=False))
print(chr(10) + 'da ghi ' + _rp)